# Machine Learning Systems for Sleep Quality Assessment Based on EEG Signals

---

* **Author**: Carmen-Theodora Craciun
* **The purpose of this Notebook**: This notebook handles ...
* **Datasets**: [Sleep-EDFX Database](https://www.physionet.org/content/sleep-edfx/1.0.0/) and [University College Dublin Sleep Apnea Database](https://physionet.org/content/ucddb/1.0.0/)
    * Sleep Cassette (SC) - the study on healthy people;
    * Sleep Telemetry (ST) - sleep study in people with difficulty falling asleep. This study was designed to look at the effects of temazepam, a drug with hypnotic effects;
    * UCDDB - patients diagnosed with apnea, who do not suffer from cardiological diseases or autonomic dysfunctions and are not taking medication at the time of registration.
* **Input**:
  * class_distribution.csv
  * results.zip - a compressed archive containing the layered and prepared dataset for training (train, test, val)
    * each file in these folders represents a *processed sleep window* (epoch)
* **Output:**
  * `model_bilstm_baseline.pt`
  * `model_crnn.pt`
  * `model_resnet.pt`

# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.exceptions import InconsistentVersionWarning
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import os
import warnings

from sklearn.metrics import accuracy_score, cohen_kappa_score
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np
import joblib
import os
from sklearn.base import clone
from tensorflow.keras import layers, models


from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore", category=InconsistentVersionWarning)
warnings.filterwarnings("ignore", message="X has feature names")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [2]:
train_path_csv = 'process_dataset/extracted_features/train_features.csv'
train_path_raw = 'process_dataset/train'
val_path_csv = 'process_dataset/extracted_features/val_features.csv'
val_path_raw = 'process_dataset/val'

class_distribution_path = 'process_dataset/class_distribution_train.csv'

# Cross Validation

In [ ]:
results =[]

In [ ]:
def cross_validation(
    model, X, y, subjects,
    model_name, model_type, save_path='./models',
    n_splits=5, sample_weights=None,
    to_calibrate=False
):
    print(f"=== Cross Validation for {model_name} ({n_splits} folds) ===")

    X_arr = np.array(X)
    y_arr = np.array(y)
    subjects_arr = np.array(subjects)

    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_accuracies = []
    fold_kappas = []

    best_accuracy = 0.0
    best_model = None

    if not os.path.exists(save_path):
        os.makedirs(save_path)

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_arr, y_arr, groups=subjects_arr), 1):

        X_train, X_val = X_arr[train_idx], X_arr[val_idx]
        y_train, y_val = y_arr[train_idx], y_arr[val_idx]

        cloned_model = clone(model)

        # 1. Train base model
        if sample_weights is None:
            cloned_model.fit(X_train, y_train)
        else:
            cloned_model.fit(X_train, y_train, sample_weight=sample_weights[train_idx])

        # 2. Optional calibration
        if to_calibrate:
            calibrated_model = CalibratedClassifierCV(
                estimator=cloned_model,
                method='sigmoid',
                cv=5
            )
            calibrated_model.fit(X_train, y_train)
            model_to_eval = calibrated_model
        else:
            model_to_eval = cloned_model

        # 3. Predict on validation
        y_pred = model_to_eval.predict(X_val)

        acc = accuracy_score(y_val, y_pred)
        kappa = cohen_kappa_score(y_val, y_pred)

        fold_accuracies.append(acc)
        fold_kappas.append(kappa)

        print(f"Fold {fold}: Accuracy = {acc:.4f}, Kappa = {kappa:.4f}")

        if acc > best_accuracy:
            best_accuracy = acc
            best_model = model_to_eval

    print("\n=== FINAL RESULTS ===")
    print(f"Mean Accuracy: {np.mean(fold_accuracies):.4f}")
    print(f"Mean Kappa: {np.mean(fold_kappas):.4f}")

    save_path_full = os.path.join(save_path, f"best_{model_name}.pkl")
    joblib.dump(best_model, save_path_full)
    print(f"Saved best model to: {save_path_full}")
    
    results_dict = {}

    results_dict[model_name] = {
        "model": model_name,
        "type": model_type,
        # "acc_mean": float(mean_acc),
        # "acc_std": float(std_acc),
        # "kappa_mean": float(mean_kappa),
        # "kappa_std": float(std_kappa),
        # "acc_best": float(best_accuracy),
        # "kappa_best": float(best_kappa),
        "fold_accs": fold_accuracies,
        "fold_kappas": fold_kappas
    }

    return results_dict

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import cohen_kappa_score, accuracy_score
import tensorflow as tf
import numpy as np
import gc
import os

def cross_validate_keras_model(
    X,
    y,
    subjects,
    build_model_fn,
    model_name,
    model_type,
    n_splits=5,
    batch_size=32,
    epochs=50,
    callbacks_list=None,
    verbose=1,
    save_dir="best_models"
):
    """
    Cross-validation CORECT pentru MLP/LSTM/CNN/ResNet
    folosind split PE SUBIECȚI (StratifiedGroupKFold).
    """

    os.makedirs(save_dir, exist_ok=True)

    X = np.array(X)
    y = np.array(y)
    subjects = np.array(subjects)

    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_results = []
    best_acc = -1
    best_model_path = None

    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, groups=subjects), 1):
        print(f"===== FOLD {fold}/{n_splits} =====")

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Detect input shape
        input_shape = X_train.shape[1:]
        num_classes = len(np.unique(y))

        # Build model
        model = build_model_fn(input_shape, num_classes)

        # Callbacks
        callbacks = callbacks_list() if callbacks_list is not None else None

        # Train
        model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=verbose
        )

        # Predict
        y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)

        acc = accuracy_score(y_val, y_pred)
        kappa = cohen_kappa_score(y_val, y_pred)

        fold_results.append((acc, kappa))

        msg = f"Accuracy = {acc:.4f}, Kappa = {kappa:.4f}"

        # Save best model
        if acc > best_acc:
            best_acc = acc
            best_model_path = os.path.join(save_dir, f"best_{model_name}.keras")
            model.save(best_model_path)
            msg += " -> Best model"

        print(msg)

        # Cleanup
        del model
        tf.keras.backend.clear_session()
        gc.collect()

    # Final results
    accs = [x[0] for x in fold_results]
    kappas = [x[1] for x in fold_results]

    print("\n===== FINAL RESULTS =====")
    print(f"Mean ACC:   {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"Mean KAPPA: {np.mean(kappas):.4f} ± {np.std(kappas):.4f}")
    print(f"Best model saved at: {best_model_path}")

    results_dict = {}

    results_dict[model_name] = {
        "model": model_name,
        "type": model_type,
        "acc_mean": float(np.mean(accs)),
        "acc_std": float(np.std(accs)),
        "kappa_mean": float(np.mean(kappas)),
        "kappa_std": float(np.std(kappas)),
        "acc_best": float(best_acc),
        "kappa_best": float(np.max(kappas)),
        "fold_accs": np.mean(accs),
        "fold_kappas": np.mean(kappas)
    }

    return results_dict

In [ ]:
import os
import shutil
import numpy as np
import tensorflow as tf
import gc
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, cohen_kappa_score

def kfold_cnn_generators_grouped(
    X_paths,y,subjects,
    build_model_fn,
    model_name,model_type,
    generator_fn,batch_size,
    epochs=40,n_splits=5,
    save_dir="best_models",
    verbose=1
):
    os.makedirs(save_dir, exist_ok=True)

    cv = GroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_results = []
    best_acc = -1
    best_model_path = None

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_paths, y, groups=subjects), 1):
        print(f"===== FOLD {fold}/{n_splits} =====")

        # 1. Directoare temporare
        fold_train_dir = f"fold_train_{fold}"
        fold_val_dir   = f"fold_val_{fold}"

        os.makedirs(fold_train_dir, exist_ok=True)
        os.makedirs(fold_val_dir, exist_ok=True)

        # 2. Copiem imaginile aferente foldului
        for idx in train_idx:
            shutil.copy(X_paths[idx], fold_train_dir)

        for idx in val_idx:
            shutil.copy(X_paths[idx], fold_val_dir)

        # 3. Generatoare
        train_gen = generator_fn(fold_train_dir, True, batch_size)
        val_gen   = generator_fn(fold_val_dir, False, batch_size)

        # 4. Model
        sample_batch, _ = train_gen[0]
        input_shape = sample_batch.shape[1:]

        model = build_model_fn(input_shape=input_shape)

        callbacks = make_callbacks()

        # 5. Train
        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=epochs,
            callbacks=callbacks,
            verbose=verbose
        )

        # 6. Evaluare
        y_true = []
        y_pred = []

        for Xb, yb in val_gen:
            y_true.extend(np.argmax(yb, axis=1))
            y_pred.extend(np.argmax(model.predict(Xb, verbose=0), axis=1))

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        acc = accuracy_score(y_true, y_pred)
        kappa = cohen_kappa_score(y_true, y_pred)

        fold_results.append((acc, kappa))

        msg = f"Accuracy = {acc:.4f}, Kappa = {kappa:.4f}"

        # 7. Salvăm cel mai bun model
        if acc > best_acc:
            best_acc = acc
            best_model_path = os.path.join(save_dir, f"best_{model_name}.keras")
            model.save(best_model_path)
            msg += " -> Best model"

        print(msg)
        print()

        # 8. Curățare memorie
        del model, train_gen, val_gen
        tf.keras.backend.clear_session()
        gc.collect()

        shutil.rmtree(fold_train_dir)
        shutil.rmtree(fold_val_dir)

    # Rezultate finale
    accs = [x[0] for x in fold_results]
    kappas = [x[1] for x in fold_results]

    print("\n===== FINAL RESULTS =====")
    print(f"Mean ACC:   {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"Mean KAPPA: {np.mean(kappas):.4f} ± {np.std(kappas):.4f}")
    print(f"\nBest model saved at: {best_model_path}")

    results_dict = {}

    results_dict[model_name] = {
        "model": model_name,
        "type": model_type,
        "acc_mean": float(np.mean(accs)),
        "acc_std": float(np.std(accs)),
        "kappa_mean": float(np.mean(kappas)),
        "kappa_std": float(np.std(kappas)),
        "acc_best": float(best_acc),
        "kappa_best": float(np.max(kappas)),
        "fold_accs": np.mean(accs),
        "fold_kappas": np.mean(kappas)
    }

    return results_dict

# Functions

In [8]:
def make_callbacks():
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=15,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_accuracy',
            factor=0.5,
            patience=5,
            min_lr=1e-5
        )
    ]

In [6]:
def build_sequences_per_subject(df, feature_cols, label_col='Label', subject_col='Subject_ID', seq_len=30, step=5):
    X_seq, y_seq, subj_seq = [], [], []
    for subj, group in df.groupby(subject_col):
        group = group.sort_index()  # Assumes chronological order; use an epoch index if available
        feats = group[feature_cols].values
        labels = group[label_col].values
        for i in range(0, len(feats) - seq_len, step):
            X_seq.append(feats[i:i+seq_len])
            y_seq.append(labels[i+seq_len-1])   # predict the last epoch
            subj_seq.append(subj)
    return np.array(X_seq), np.array(y_seq), np.array(subj_seq)

In [7]:
def se_feature_attention(inputs, reduction=8):
    n = inputs.shape[-1]

    x = layers.Dense(n // reduction, activation='relu')(inputs)
    x = layers.Dense(n, activation='sigmoid')(x)

    return layers.Multiply()([inputs, x])

In [8]:
def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_true_one_hot = tf.one_hot(tf.cast(y_true, tf.int32), depth=5)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        ce_term = -y_true_one_hot * tf.math.log(y_pred)
        p_t = tf.reduce_sum(y_true_one_hot * y_pred, axis=-1)
        focal_modulator = tf.pow(1. - p_t, gamma)
        focal_loss_per_class = tf.expand_dims(focal_modulator, axis=-1) * ce_term
        return tf.reduce_mean(tf.reduce_sum(alpha * focal_loss_per_class, axis=-1))
    return loss

In [ ]:
from scipy.signal import resample

class SleepDataGenerator(tf.keras.utils.PyDataset):
    def __init__(self,
                 directory,
                 is_training=True,
                 batch_size=32,
                 n_classes=5,
                 window_size=3,
                 use_context=False,
                 use_lag_lead=False,
                 downsample=False,
                 target_length=1500,
                 sampling_rate=50,
                 lag_seconds=0.5,
                 **kwargs):
        super().__init__()
        self.directory = directory
        self.batch_size = batch_size
        self.n_classes = n_classes
        self.is_training = is_training
        self.file_list = [f for f in os.listdir(directory) if f.endswith('.npz')]
        self.on_epoch_end()

        self.use_context = use_context
        self.use_lag_lead = use_lag_lead
        self.downsample = downsample
        self.target_length = target_length
        self.sampling_rate = sampling_rate
        self.lag_steps = int(lag_seconds * sampling_rate)

        if self.use_context:
            self.window_size = window_size
        else:
            self.window_size = 1

    def __len__(self):
        return len(self.file_list)

    def _normalize(self, X_batch):
        # normalize PER EPOCH, pe axa timpului
        mean = np.mean(X_batch, axis=1, keepdims=True)
        std  = np.std(X_batch, axis=1, keepdims=True)
        return (X_batch - mean) / (std + 1e-8)

    def _add_lag_lead(self, X_batch):
        if self.use_lag_lead:
            # lag/lead în TIMP, nu 1 sample
            lag  = np.roll(X_batch,  self.lag_steps,  axis=1)
            lead = np.roll(X_batch, -self.lag_steps, axis=1)
            X_batch = np.concatenate([X_batch, lag, lead], axis=-1)
        return X_batch

    def __getitem__(self, index):
        file_path = os.path.join(self.directory, self.file_list[index])
        with np.load(file_path, allow_pickle=True) as data:
            X_raw = data['x'] if 'x' in data else data['arr_0']
            y_raw = data['y'] if 'y' in data else data['arr_1']

        X = X_raw.copy()
        y = y_raw.copy()

        if len(X.shape) == 3 and X.shape[-1] == 3000 and X.shape[1] == 2:
            X = np.transpose(X, (0, 2, 1))

        if self.downsample:
            X_resampled = np.zeros((X.shape[0], self.target_length, X.shape[2]), dtype=X.dtype)
            for i in range(X.shape[0]):
                for ch in range(X.shape[2]):
                    X_resampled[i, :, ch] = resample(X[i, :, ch], self.target_length)
            X = X_resampled

        if self.use_context:
            return self._get_context_batch(X, y)
        else:
            return self._get_single_batch(X, y)

    def _get_single_batch(self, X, y):
        if self.is_training:
            idx = np.random.choice(X.shape[0], min(self.batch_size, X.shape[0]), replace=False)
            X_batch = X[idx]
            y_batch = y[idx]
        else:
            X_batch = X
            y_batch = y

        X_batch = self._normalize(X_batch)
        X_batch = self._add_lag_lead(X_batch)
        return X_batch, tf.keras.utils.to_categorical(y_batch, num_classes=self.n_classes)

    def _get_context_batch(self, X, y):
        num_epochs = X.shape[0]
        X_windows, y_windows = [], []

        for i in range(self.window_size - 1, num_epochs):
            window = X[i - self.window_size + 1 : i + 1]   # (window_size, time, ch)
            window_concat = np.concatenate(window, axis=0) # (window_size*time, ch)
            X_windows.append(window_concat)
            y_windows.append(y[i])

        X_windows = np.array(X_windows)
        y_windows = np.array(y_windows)

        if self.is_training:
            idx = np.random.choice(X_windows.shape[0], min(self.batch_size, X_windows.shape[0]), replace=False)
            X_batch = X_windows[idx]
            y_batch = y_windows[idx]
        else:
            X_batch = X_windows
            y_batch = y_windows

        X_batch = self._normalize(X_batch)
        X_batch = self._add_lag_lead(X_batch)
        return X_batch, tf.keras.utils.to_categorical(y_batch, num_classes=self.n_classes)

    def on_epoch_end(self):
        if self.is_training:
            np.random.shuffle(self.file_list)

# Dataset

In [10]:
import pandas as pd

df_train = pd.read_csv(train_path_csv)
df_val = pd.read_csv(val_path_csv)
df_total = pd.concat([df_train, df_val], axis=0)

subjects = df_total['Subject_ID'].values

cols_to_drop = ['Subject_ID', 'Dataset_Source', 'Label']
feature_cols = [c for c in df_total.columns if c not in cols_to_drop]

X_total = df_total[feature_cols]
y_total = df_total['Label']

scaler = StandardScaler()
X_total_scaled = scaler.fit_transform(X_total.fillna(0))

X_total_seq, y_total_seq, subjects_seq = build_sequences_per_subject(df_total, feature_cols, seq_len=30, step=5)

In [11]:
class_distribution = pd.read_csv(class_distribution_path)

class_weights = dict(zip(class_distribution['Class_ID'], class_distribution['Weight']))

sample_weights_custom = np.array([class_weights[y] for y in y_total])

# Models Cross Validation

In [12]:
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")


In [13]:
results = []

## LightGBM

In [50]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.08,
    max_depth=10,
    num_leaves=125,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)


result = cross_validation(
    model=lgb_model,
    X=X_total,
    y=y_total,
    subjects=subjects,
    model_name="LGBM_calibrated",
    model_type='Tree-Based',
    sample_weights=sample_weights_custom,
    save_path="./models",
    to_calibrate=True
)


results.append(result)

=== Cross Validation for LGBM_calibrated (5 folds) ===
Fold 1: Accuracy = 0.7822, Kappa = 0.7029
Fold 2: Accuracy = 0.7666, Kappa = 0.6816
Fold 3: Accuracy = 0.7568, Kappa = 0.6683
Fold 4: Accuracy = 0.7642, Kappa = 0.6797
Fold 5: Accuracy = 0.7858, Kappa = 0.7113

=== FINAL RESULTS ===
Mean Accuracy: 0.7711
Mean Kappa: 0.6888
Saved best model to: ./models/best_LGBM_calibrated.pkl


## XGBoost

In [21]:
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    eval_metric='mlogloss',
    random_state=42,
    n_estimators=400,
    learning_rate=0.08,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    enable_categorical=True,
    n_jobs=-1
)

result = cross_validation(
    model=xgb_model,
    X=X_total,
    y=y_total,
    subjects=subjects,
    model_name="XGB_calibrated",
    model_type='Tree-Based',
    sample_weights=sample_weights_custom,
    save_path="./models",
    to_calibrate=True
)

results.append(result)

=== Cross Validation for XGB_calibrated (5 folds) ===
Fold 1: Accuracy = 0.7802, Kappa = 0.7003
Fold 2: Accuracy = 0.7679, Kappa = 0.6838
Fold 3: Accuracy = 0.7565, Kappa = 0.6681
Fold 4: Accuracy = 0.7636, Kappa = 0.6793
Fold 5: Accuracy = 0.7836, Kappa = 0.7086

=== FINAL RESULTS ===
Mean Accuracy: 0.7704
Mean Kappa: 0.6880
Saved best model to: ./models/best_XGB_calibrated.pkl


## Random Forest

In [22]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

result = cross_validation(
    model=rf_model,
    X=X_total,
    y=y_total,
    subjects=subjects,
    model_name="RF_calibrated",
    model_type='Tree-Based',
    sample_weights=sample_weights_custom,
    save_path="./models",
    to_calibrate=True
)

results.append(result)

=== Cross Validation for RF_calibrated (5 folds) ===
Fold 1: Accuracy = 0.7524, Kappa = 0.6625
Fold 2: Accuracy = 0.7389, Kappa = 0.6433
Fold 3: Accuracy = 0.7308, Kappa = 0.6330
Fold 4: Accuracy = 0.7340, Kappa = 0.6386
Fold 5: Accuracy = 0.7602, Kappa = 0.6763

=== FINAL RESULTS ===
Mean Accuracy: 0.7433
Mean Kappa: 0.6507
Saved best model to: ./models/best_RF_calibrated.pkl


## LDA

In [23]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression

lda_lsqr = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')

result = cross_validation(
    model=lda_lsqr,
    X=X_total_scaled,
    y=y_total,
    subjects=subjects,
    model_name="LDA-Baseline",
    model_type='Probabilistic',
    save_path="./models"
)

results.append(result)

=== Cross Validation for LDA-Baseline (5 folds) ===
Fold 1: Accuracy = 0.7391, Kappa = 0.6459
Fold 2: Accuracy = 0.7234, Kappa = 0.6239
Fold 3: Accuracy = 0.7122, Kappa = 0.6098
Fold 4: Accuracy = 0.7134, Kappa = 0.6116
Fold 5: Accuracy = 0.7377, Kappa = 0.6467

=== FINAL RESULTS ===
Mean Accuracy: 0.7251
Mean Kappa: 0.6276
Saved best model to: ./models/best_LDA-Baseline.pkl


## Logistic Regression

In [25]:
lr_lbfgs = LogisticRegression(
    class_weight='balanced',
    max_iter=5000,
    C=0.5,
    solver='lbfgs',
    random_state=42
)

result = cross_validation(
    model=lr_lbfgs,
    X=X_total_scaled,
    y=y_total,
    subjects=subjects,
    model_name="LogisticR-Baseline",
    model_type='Probabilistic',
    save_path="./models"
)

results.append(result)

=== Cross Validation for LogisticR-Baseline (5 folds) ===
Fold 1: Accuracy = 0.7299, Kappa = 0.6477
Fold 2: Accuracy = 0.7033, Kappa = 0.6137
Fold 3: Accuracy = 0.6939, Kappa = 0.6031
Fold 4: Accuracy = 0.7116, Kappa = 0.6239
Fold 5: Accuracy = 0.7218, Kappa = 0.6407

=== FINAL RESULTS ===
Mean Accuracy: 0.7121
Mean Kappa: 0.6258
Saved best model to: ./models/best_LogisticR-Baseline.pkl


## MLP

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

def build_mlp(input_shape, n_classes, dropout_rate=0.3):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        layers.Dense(64, activation='relu'),
        layers.Dense(n_classes, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [44]:
result = cross_validate_keras_model(
    X=X_total,
    y=y_total,
    subjects=subjects,
    build_model_fn=build_mlp,
    model_name="mlp",
    model_type='ANN',
    batch_size=32,
    epochs=50,
    callbacks_list=make_callbacks,
    save_dir="./models"
)

results.append(result)

===== FOLD 1/5 =====
Epoch 1/50
7002/7002 ━━━━━━━━━━━━━━━━━━━━ 31s 4ms/step - accuracy: 0.7177 - loss: 0.7441 - val_accuracy: 0.7570 - val_loss: 0.6392 - learning_rate: 0.0010
Epoch 2/50
7002/7002 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms/step - accuracy: 0.7449 - loss: 0.6675 - val_accuracy: 0.7617 - val_loss: 0.6266 - learning_rate: 0.0010
Epoch 3/50
7002/7002 ━━━━━━━━━━━━━━━━━━━━ 29s 4ms/step - accuracy: 0.7546 - loss: 0.6405 - val_accuracy: 0.7693 - val_loss: 0.6069 - learning_rate: 0.0010
Epoch 4/50
7002/7002 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms/step - accuracy: 0.7601 - loss: 0.6240 - val_accuracy: 0.7698 - val_loss: 0.5992 - learning_rate: 0.0010
Epoch 5/50
7002/7002 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms/step - accuracy: 0.7662 - loss: 0.6101 - val_accuracy: 0.7735 - val_loss: 0.5956 - learning_rate: 0.0010
Epoch 6/50
7002/7002 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms/step - accuracy: 0.7688 - loss: 0.6029 - val_accuracy: 0.7755 - val_loss: 0.5946 - learning_rate: 0.0010
Epoch 7/50
7002/7002 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms

## LSTM

In [ ]:
def build_lstm_se_focal(input_shape, n_classes, units=128, dropout=0.4, recurrent_dropout=0.2):
    inputs = layers.Input(shape=input_shape)

    x = layers.LSTM(
        units,
        return_sequences=True,
        dropout=dropout,
        recurrent_dropout=recurrent_dropout
    )(inputs)
    x = layers.BatchNormalization()(x)

    x = layers.LSTM(
        units // 2,
        dropout=dropout,
        recurrent_dropout=recurrent_dropout
    )(x)
    x = layers.BatchNormalization()(x)

    x = se_feature_attention(x)

    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer='l2')(x)
    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer='adam',
        loss=focal_loss(gamma=2.0),
        metrics=['accuracy']
    )
    return model

In [15]:
result = cross_validate_keras_model(
    X=X_total_seq,
    y=y_total_seq,
    subjects=subjects_seq,
    build_model_fn=build_lstm_se_focal,
    model_name="lstm_se_focal",
    model_type='ANN',
    batch_size=32,
    epochs=50,
    callbacks_list=make_callbacks,
    save_dir="./models"
)

results.append(result)

===== FOLD 1/5 =====


E0000 00:00:1779525908.822190  335827 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 98s 68ms/step - accuracy: 0.6523 - loss: 0.1707 - val_accuracy: 0.7169 - val_loss: 0.0981 - learning_rate: 0.0010
Epoch 2/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 91s 67ms/step - accuracy: 0.7214 - loss: 0.0942 - val_accuracy: 0.7315 - val_loss: 0.0873 - learning_rate: 0.0010
Epoch 3/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 90s 66ms/step - accuracy: 0.7354 - loss: 0.0866 - val_accuracy: 0.7416 - val_loss: 0.0828 - learning_rate: 0.0010
Epoch 4/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 90s 66ms/step - accuracy: 0.7485 - loss: 0.0814 - val_accuracy: 0.7500 - val_loss: 0.0818 - learning_rate: 0.0010
Epoch 5/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 90s 66ms/step - accuracy: 0.7545 - loss: 0.0780 - val_accuracy: 0.7458 - val_loss: 0.0847 - learning_rate: 0.0010
Epoch 6/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 91s 67ms/step - accuracy: 0.7584 - loss: 0.0757 - val_accuracy: 0.7544 - val_loss: 0.0852 - learning_rate: 0.0010
Epoch 7/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 90s 66ms/step - accura

## ResNet

In [16]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_resnet_feature_model(seq_len, n_features, n_classes=5):

    inp = layers.Input(shape=(seq_len, n_features))

    def conv_block(x, filters):
        shortcut = x

        x = layers.Conv1D(filters, 3, padding='same',
                          kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)

        x = layers.Conv1D(filters, 3, padding='same',
                          kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)

        if shortcut.shape[-1] != filters:
            shortcut = layers.Conv1D(filters, 1, padding='same',
                                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(shortcut)

        x = layers.Add()([x, shortcut])
        x = layers.ReLU()(x)
        x = layers.Dropout(0.2)(x)
        return x

    x = conv_block(inp, 32)
    x = conv_block(x, 64)
    x = conv_block(x, 128)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(3e-4),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )
    return model

def build_resnet_wrapper(input_shape,num_classes=5):
    seq_len = input_shape[0]
    n_features = input_shape[1]
    return build_resnet_feature_model(seq_len, n_features, num_classes)

In [17]:
result = cross_validate_keras_model(
    X=X_total_seq,
    y=y_total_seq,
    subjects=subjects_seq,
    build_model_fn=build_resnet_wrapper,
    model_name="resnet_feature",
    model_type='ANN',
    batch_size=32,
    epochs=50,
    callbacks_list=make_callbacks,
    save_dir="./models"
)

results.append(result)

===== FOLD 1/5 =====
Epoch 1/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 47s 31ms/step - accuracy: 0.6276 - loss: 1.0436 - val_accuracy: 0.7051 - val_loss: 0.8371 - learning_rate: 3.0000e-04
Epoch 2/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 40s 29ms/step - accuracy: 0.7157 - loss: 0.8226 - val_accuracy: 0.7274 - val_loss: 0.7938 - learning_rate: 3.0000e-04
Epoch 3/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 40s 30ms/step - accuracy: 0.7448 - loss: 0.7439 - val_accuracy: 0.7439 - val_loss: 0.7349 - learning_rate: 3.0000e-04
Epoch 4/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 41s 30ms/step - accuracy: 0.7639 - loss: 0.6972 - val_accuracy: 0.7565 - val_loss: 0.7191 - learning_rate: 3.0000e-04
Epoch 5/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 40s 29ms/step - accuracy: 0.7719 - loss: 0.6700 - val_accuracy: 0.7550 - val_loss: 0.7207 - learning_rate: 3.0000e-04
Epoch 6/50
1365/1365 ━━━━━━━━━━━━━━━━━━━━ 40s 29ms/step - accuracy: 0.7779 - loss: 0.6509 - val_accuracy: 0.7588 - val_loss: 0.7130 - learning_rate: 3.0000e-04
Epoch 7/50
1365/136